## Step 1) Install libraries

In [1]:
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.core.chat_engine import ContextChatEngine
from llama_index.core.memory import ChatMemoryBuffer
from llama_index.core.base.llms.types import ChatMessage, MessageRole

c:\Users\sanaz\miniconda3\envs\llama\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2) Load API keys

In [ ]:
# Load API key from .env
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("GROQ_API_K")) 

: 

## Step 3) Set up the LLM

In [3]:
# Import Groq LLM from LlamaIndex
from llama_index.llms.groq import Groq

# Choose the LLM model
model = "llama-3.3-70b-versatile"

# Create the LLM object
llm = Groq(model=model)

## Step 4) Set up the embedding model

In [4]:
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbedding(
    model_name=embedding_model_name,
    cache_folder="/content/embedding_model"
)

## Step 5) Build vector index from your PDFs

In [5]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    "./data",
    required_exts=[".pdf"]
).load_data(show_progress=True)

print(f"Loaded {len(documents)} documents.")

Loading files: 100%|██████████| 1/1 [00:03<00:00,  3.38s/it]

Loaded 36 documents.


In [6]:
from llama_index.core.node_parser import SentenceSplitter

text_splitter = SentenceSplitter(
    chunk_size=800,
    chunk_overlap=150
)

In [7]:
# ===== Build vector index from your PDFs =====
vector_index = VectorStoreIndex.from_documents(
    documents,
    transformations=[text_splitter],
    embed_model=embeddings
)

## Step 6) Retriever 

In [8]:
retriever = vector_index.as_retriever(similarity_top_k=2)



## Step 7) prefix_messages

In [9]:
prefix_messages = [
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="You are a nice chatbot having a conversation with a human.",
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="Answer the question based only on the following context and previous conversation.",
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content='If the answer is not found in the document, say: "The answer is not found in the document."',
    ),
    ChatMessage(
        role=MessageRole.SYSTEM,
        content="Keep your answers short and succinct.",
    ),
]

## Step 8) Memory

In [10]:
memory = ChatMemoryBuffer.from_defaults()

## Step 9) RAG chatbot with memory

In [11]:
# ===== RAG chatbot with memory =====
rag_bot = ContextChatEngine(
    llm=llm,
    retriever=retriever,
    memory=memory,
    prefix_messages=prefix_messages,
)

In [12]:
while True:
    user_input = input("You: ").strip()

    if user_input.lower() == "end":
        print("Ending the conversation. Goodbye!")
        break

    if not user_input:
        print("Please enter a valid question.")
        print("-" * 50)
        continue

    response = rag_bot.chat(user_input)

    print("\nQuestion:")
    print(user_input)

    print("\nAnswer:")
    print(response.response)
    print("-" * 50)

2026-03-29 19:42:55,957 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



Question:
what is green house gas?

Answer:
The document doesn't explicitly define what a greenhouse gas is, but it mentions examples of greenhouse gases, including carbon dioxide, methane, and nitrous oxide.
--------------------------------------------------
Ending the conversation. Goodbye!
